# IMPORTS

In [4]:
import os
import shutil
import subprocess
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta, timezone
from pathlib import Path
from torch.utils.data import DataLoader, random_split
from torchvision import transforms, models, datasets
from torchvision.models import EfficientNet_V2_S_Weights
from tqdm.auto import tqdm
from PIL import Image
from IPython import get_ipython
from sklearn.metrics import (confusion_matrix, classification_report, roc_curve,
                             auc, precision_recall_curve, average_precision_score, accuracy_score)
from sklearn.preprocessing import label_binarize
from scipy.signal import savgol_filter
import kagglehub
import gc
import sys
import inspect
import pprint
from matplotlib.gridspec import GridSpec
from matplotlib.collections import LineCollection
from matplotlib.ticker import MaxNLocator
import stat
if 'google.colab' in sys.modules:
    print("Running in Google Colab")
    is_colab = True
    from google.colab import _message
    from google.colab import userdata
else:
    print("Not running in Google Colab")
    is_colab = False
    torch.hub.set_dir('weights')
from imagededup.methods import PHash

Not running in Google Colab


# ДАННЫЕ

In [5]:
if is_colab:
    token = userdata.get('GITHUB_DLTESTWORK_TOKEN')
else:
    token = os.environ.get('GITHUB_DLTESTWORK_TOKEN')
    os.environ["KAGGLEHUB_CACHE"] = "datasets"
path_filtered = kagglehub.dataset_download("kvdep1/dumplings")
path_sub = kagglehub.competition_download("yummi-classification-fu25")

In [6]:


def clean_duplicates(image_dir, threshold=10):
    print(f"Scanning for duplicates in {image_dir}...")
    phasher = PHash()
    
    # Generate encodings for all images
    # recursive=True ensures we scan subfolders (classes)
    encodings = phasher.encode_images(image_dir=image_dir, recursive=True)
    
    # Find duplicates
    # max_distance_threshold: 0 is exact, 10 is very similar
    duplicates = phasher.find_duplicates(encoding_map=encodings, max_distance_threshold=threshold)
    
    files_to_remove = set()
    
    for filename, duplicate_list in duplicates.items():
        if filename in files_to_remove:
            continue
            
        # If this file has duplicates, mark the duplicates for removal
        # We keep the 'filename' (original) and remove the others
        for dup in duplicate_list:
            files_to_remove.add(dup)
            
    print(f"Found {len(files_to_remove)} duplicates to remove.")
    
    # Remove files
    for f in files_to_remove:
        path_to_remove = Path(image_dir) / f
        if path_to_remove.exists():
            try:
                os.remove(path_to_remove)
            except Exception as e:
                print(f"Error removing {f}: {e}")
                
    print("Cleaning finished.")



# LOGGING

In [7]:
class GitLogger:
    def __init__(self, repo_url, token, base_folder="dumplings"):
        # 1. Config
        self.user_email = os.environ.get('GIT_EMAIL') or 'bot@experiment.com'
        self.user_name = os.environ.get('GIT_NAME') or 'Experiment Bot'
        self.base_folder = base_folder
        
        # 2. Setup Auth URL
        token = str(token).strip()
        repo_url = repo_url.strip().replace("https://", "").replace("http://", "")
        self.auth_url = f"https://{token}@{repo_url}"
        self.repo_name = repo_url.split("/")[-1].replace(".git", "")
        self.local_path = os.path.join(os.getcwd(), self.repo_name)
        
        # 3. Initialize (Metadata Only - No File Download)
        self._init_repo_safe()
        
        # 4. Create NEW Experiment Folder (Safe Name)
        # Naming: dumplings/0p85_2025-12-18_13-00
        timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
        self.exp_name = timestamp # Will append metric later if needed, or use timestamp now
        self.exp_path = os.path.join(self.local_path, self.base_folder, self.exp_name)
        
        os.makedirs(self.exp_path, exist_ok=True)
        print(f"[GitLogger] Ready. Target: {self.exp_path}")

    def _init_repo_safe(self):
        """
        Clones Git History ONLY. 
        Robustly clears previous folder even on Windows.
        """
        # --- 1. Robust Folder Deletion ---
        if os.path.exists(self.local_path):
            print(f"[GitLogger] Cleaning up old directory: {self.local_path}")
            
            # Windows Helper: Unset read-only bit to allow deletion
            def on_rm_error(func, path, exc_info):
                os.chmod(path, stat.S_IWRITE)
                func(path)

            try:
                shutil.rmtree(self.local_path, onerror=on_rm_error)
            except Exception as e:
                print(f"[GitLogger] Warning: Could not delete folder directly: {e}")
                # Fallback: Rename it out of the way if delete fails (e.g. file locked)
                timestamp = datetime.now().strftime("%H%M%S")
                os.rename(self.local_path, f"{self.local_path}_trash_{timestamp}")

        # Double check it is gone
        if os.path.exists(self.local_path):
             raise RuntimeError(f"Cannot clear target folder '{self.local_path}'. Close open files/terminals.")

        # --- 2. Clone ---
        print(f"[GitLogger] Cloning metadata only (Safe Mode)...")
        try:
            # Added capture_output=True to see errors if they happen again
            subprocess.run([
                "git", "clone", 
                "--filter=blob:none", 
                "--no-checkout",       
                "--depth=1",           
                self.auth_url, 
                self.local_path
            ], check=True, capture_output=True)
            
            # Configure Git
            subprocess.run(["git", "-C", self.local_path, "config", "user.email", self.user_email], check=False)
            subprocess.run(["git", "-C", self.local_path, "config", "user.name", self.user_name], check=False)
            
            # Reset Index
            subprocess.run(["git", "-C", self.local_path, "reset", "--mixed", "HEAD"], check=True, capture_output=True)
            
        except subprocess.CalledProcessError as e:
            # Print the actual error from Git
            err_msg = e.stderr.decode('utf-8') if e.stderr else "Unknown error"
            print(f"GIT ERROR: {err_msg}")
            raise RuntimeError(f"Git Init Failed. Details: {err_msg}")
    
    def start_experiment(self, metric=None):
        # Optional: Rename folder to include metric now that we have it
        if metric is not None:
            safe_metric = f"{metric:.4f}"#.replace(".", "p")
            new_name = f"{safe_metric}_{self.exp_name}"
            new_path = os.path.join(self.local_path, self.base_folder, new_name)
            
            # Rename directory
            os.rename(self.exp_path, new_path)
            self.exp_path = new_path
            self.exp_name = new_name
            print(f"[GitLogger] Updated Folder Name: {self.exp_name}")

        with open(os.path.join(self.exp_path, "info.txt"), "w") as f:
            f.write(f"Started. Metric: {metric}")

    def push_results(self, commit_msg="Auto-log"):
        print("[GitLogger] Pushing new data...")
        try:
            # 1. Add ONLY the new folder (Explicit path to avoid adding 'deleted' files from missing checkout)
            # We use '/' for git compatibility even on Windows
            target_rel_path = f"{self.base_folder}/{self.exp_name}"
            
            subprocess.run(["git", "-C", self.local_path, "add", target_rel_path], check=True)
            
            # 2. Commit
            subprocess.run(["git", "-C", self.local_path, "commit", "-m", f"{commit_msg} {self.exp_name}"], check=True)
            
            # 3. Push
            subprocess.run(["git", "-C", self.local_path, "push"], check=True)
            print("[GitLogger] Success.")
        except subprocess.CalledProcessError as e:
            print(f"[GitLogger] Push failed: {e}")

    # --- Helpers ---
    def save_code(self):
        dest = os.path.join(self.exp_path, "run_code.py")
        if os.path.exists(sys.argv[0]): shutil.copy2(sys.argv[0], dest)
        else: open(dest, "w").write("# Code not found")

    def save_csv(self, df, name):
        df.to_csv(os.path.join(self.exp_path, name), index=False)
        
    def save_figure(self, fig, name):
        fig.savefig(os.path.join(self.exp_path, name))

    def log_result(self, metrics, commit_msg="Log"):
        with open(os.path.join(self.exp_path, "metrics.txt"), "w") as f:
            for k, v in metrics.items(): f.write(f"{k}: {v}\n")
        self.save_code()
        self.push_results(commit_msg)


# DATA + MODEL

In [8]:
if 'is_colab' not in globals():
    is_colab = 'google.colab' in sys.modules
if is_colab:
    def to_rgb(img):
        return img.convert('RGB')

    def get_dataloaders(CONFIG):
        # Размер изображения из конфига
        img_size = CONFIG['img_size']

        # 1. Аугментации для ТРЕНИРОВКИ
        train_tf = transforms.Compose([
            # FIX: Use the global function instead of lambda
            transforms.Lambda(to_rgb),

            # 2. Геометрия
            transforms.RandomResizedCrop(img_size, scale=(0.5, 1.0), ratio=(0.75, 1.33)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.3),
            transforms.RandomRotation(degrees=45),

            # Перспектива
            transforms.RandomPerspective(distortion_scale=0.4, p=0.4),
            transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), shear=15),

            # 3. Качество фото
            transforms.RandomApply([
                transforms.GaussianBlur(kernel_size=(5, 9), sigma=(0.1, 5))
            ], p=0.3),

            transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
            transforms.RandomGrayscale(p=0.1),

            # 4. Тензоры
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),

            # 5. Cutout
            transforms.RandomErasing(p=0.3, scale=(0.02, 0.2), ratio=(0.3, 3.3))
        ])

        # Валидация
        val_tf = transforms.Compose([
            # FIX: Use the global function here too
            transforms.Lambda(to_rgb),
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])

        full_data_train = datasets.ImageFolder(CONFIG['data_dir'], transform=train_tf)
        full_data_val   = datasets.ImageFolder(CONFIG['data_dir'], transform=val_tf)

        ds_len = len(full_data_train)
        train_sz = int(0.8 * ds_len)
        val_sz = ds_len - train_sz

        generator = torch.Generator().manual_seed(42)
        train_ds, _ = random_split(full_data_train, [train_sz, val_sz], generator=generator)
        _, val_ds   = random_split(full_data_val,   [train_sz, val_sz], generator=generator)

        # Now num_workers=2 will work on Windows
        if is_colab:
            workers=2
        else:
            workers=0
        train_dl = DataLoader(train_ds, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=workers, pin_memory=True)
        val_dl   = DataLoader(val_ds, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=workers, pin_memory=True)

        return train_dl, val_dl, full_data_train.classes
    def build_model(num_classes):
        weights = EfficientNet_V2_S_Weights.DEFAULT
        model = models.efficientnet_v2_s(weights=weights)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
        return model.to(CONFIG['device'])
else:
    from dataloaders import get_dataloaders

def build_model(num_classes):
    weights = EfficientNet_V2_S_Weights.DEFAULT
    model = models.efficientnet_v2_s(weights=weights)
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    return model.to(CONFIG['device'])

# TRAIN

In [9]:
def train(CONFIG=None):
    train_loader, val_loader, classes = get_dataloaders(CONFIG)
    model = build_model(len(classes))
    optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG['lr'])
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=CONFIG['lr'],
                                                    steps_per_epoch=len(train_loader), epochs=CONFIG['epochs'])
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    history = {'train_loss': [], 'val_acc': []}
    best_acc = 0.0

    print("Starting Training...")
    for epoch in range(CONFIG['epochs']):
        model.train()
        epoch_loss = 0
        for x, y in tqdm(train_loader, desc=f"Epoch {epoch+1}", leave=False):
            x, y = x.to(CONFIG['device']), y.to(CONFIG['device'])
            optimizer.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            optimizer.step()
            scheduler.step()
            epoch_loss += loss.item()

        avg_loss = epoch_loss / len(train_loader)
        history['train_loss'].append(avg_loss)

        # Validation
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(CONFIG['device']), y.to(CONFIG['device'])
                out = model(x)
                _, pred = torch.max(out, 1)
                total += y.size(0)
                correct += (pred == y).sum().item()

        acc = correct / total
        history['val_acc'].append(acc)
        print(f"Epoch {epoch+1} | Loss: {avg_loss:.4f} | Val Acc: {acc:.4f}")

        if acc > best_acc:
            best_acc = acc
            torch.save(model.state_dict(), "best_model.pth")

    return model, val_loader, classes, history, best_acc

# VIS

In [10]:
def run_full_evaluation(model, loader, logger, history, classes, config):
    print("Running Full Evaluation...")
    model.eval()
    y_true, y_pred, y_prob = [], [], []

    # 1. Inference
    with torch.no_grad():
        for x, y in loader:
            x = x.to(config['device'])
            out = model(x)
            prob = torch.nn.functional.softmax(out, dim=1)
            y_true.extend(y.cpu().numpy())
            y_pred.extend(torch.max(prob, 1)[1].cpu().numpy())
            y_prob.extend(prob.cpu().numpy())

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    y_prob = np.array(y_prob)
    n_classes = len(classes)

    # --- Styling & Colors ---
    MAIN_RED = '#C62828'
    CLS_COLORS = ['#C62828', '#1565C0', '#F9A825', '#2E7D32']

    # Global Font Sizing
    plt.rcParams.update({
        'font.family': 'sans-serif',
        'font.weight': 'bold',
        'font.size': 14,
        'axes.titlesize': 26,
        'axes.labelsize': 18,
        'xtick.labelsize': 14,
        'ytick.labelsize': 14,
        'legend.fontsize': 14,
        'figure.facecolor': 'white',
        'axes.facecolor': 'white',
        'text.color': 'black',
        'axes.labelcolor': 'black',
        'axes.edgecolor': '#333333',
        'axes.grid': True,
        'grid.alpha': 0.2,
        'grid.color': 'black'
    })

    # Layout: 3 Rows, 3 Columns
    fig = plt.figure(figsize=(24, 20), constrained_layout=True)
    # Column width ratios: Middle slightly thinner for the tall table
    gs = fig.add_gridspec(3, 3, height_ratios=[1, 1, 1], width_ratios=[0.9, 1.2, 1])

    # Time (Moscow +3)
    moscow_time = datetime.now(timezone.utc) + timedelta(hours=3)
    fig.suptitle(f"Evaluation Report | {moscow_time.strftime('%Y-%m-%d %H:%M')}",
                 fontsize=48, y=1.3, fontweight='bold')

    # --- Helper: Gradient Line Plot ---
    def plot_gradient_line(ax, x, y, title, ylabel=None, rising_is_good=True, is_loss=False):
        # Strict Y Limits 0-1 (Only if not loss)
        if not(is_loss): ax.set_ylim(0, 1.02)

        # Strict X Limits 0 to N+1
        max_epoch = max(x) if len(x) > 0 else 1
        ax.set_xlim(0, max_epoch + 1)
        ax.set_xticks(np.arange(1, max_epoch + 1))

        if len(x) < 2:
            # If only 1 point, just draw the dot
            val = y[0] if len(y) > 0 else 0
            best_idx = 0
        else:
            points = np.array([x, y]).T.reshape(-1, 1, 2)
            segments = np.concatenate([points[:-1], points[1:]], axis=1)
            diffs = np.diff(y)
            # Determine colors based on slope
            colors = [MAIN_RED if (d >= 0 if rising_is_good else d <= 0) else 'black' for d in diffs]

            lc = LineCollection(segments, colors=colors, linewidths=5)
            ax.add_collection(lc)

            # Find the Best Point (Max for Acc, Min for Loss)
            if is_loss:
                best_idx = np.argmin(y)
            else:
                best_idx = np.argmax(y)

        # Use MAIN_RED for the highlighted best point
        target_x = x[best_idx]
        target_y = y[best_idx]

        ax.scatter(target_x, target_y, color='black', s=200, zorder=5)

        # --- FIX: Black Text & Closer to Dot ---
        # We use annotate with 'offset points' to place text 15 pixels above the dot
        # This handles both Loss (small scale) and Acc (0-1 scale) consistently.
        ax.annotate(f"{target_y:.4f}",
                    (target_x, target_y),
                    textcoords="offset points",
                    xytext=(0, 15), # 15 points vertically up
                    ha='center',
                    color='black',  # Changed from MAIN_RED to black
                    fontweight='bold',
                    fontsize=20)

        ax.set_title(title, pad=20, fontsize=36, fontweight='bold')
        if ylabel: ax.set_ylabel(ylabel)


    # A. Accuracy (Row 0, Cols 0-1)
    ax_acc = fig.add_subplot(gs[0, :2])
    epochs = np.arange(1, len(history['val_acc']) + 1)
    plot_gradient_line(ax_acc, epochs, history['val_acc'], 'Accuracy', rising_is_good=True)

    # B. ROC Curves (Row 0, Col 2)
    ax_roc = fig.add_subplot(gs[0, 2])
    y_bin = label_binarize(y_true, classes=range(n_classes))
    for i in range(n_classes):
        if np.sum(y_bin[:, i]) > 0:
            fpr, tpr, _ = roc_curve(y_bin[:, i], y_prob[:, i])
            roc_auc = auc(fpr, tpr)
            ax_roc.plot(fpr, tpr, lw=4, color=CLS_COLORS[i % 4],
                        label=f'{classes[i]} ({roc_auc:.2f})')
    ax_roc.plot([0, 1], [0, 1], 'k--', lw=2, alpha=0.4)
    ax_roc.set_xlim(-0.02, 1.02)
    ax_roc.set_ylim(-0.02, 1.02)
    ax_roc.set_title('ROC Curves', fontsize=36, fontweight='bold')
    ax_roc.legend(loc='lower right', frameon=False, fontsize=25)

    # C. Confusion Matrix (Row 1, Col 0)
    ax_cm = fig.add_subplot(gs[1, 0])
    cm = confusion_matrix(y_true, y_pred)
    cmap_rb = sns.blend_palette(['white', '#dddddd', 'black', MAIN_RED], as_cmap=True)
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap_rb, ax=ax_cm, cbar=False,
                xticklabels=classes, yticklabels=classes, square=True,
                annot_kws={"size": 24, "weight": "bold", "color": "white"})
    # Manual color fix for annotations
    for text in ax_cm.texts:
        if int(text.get_text()) > cm.max() / 2: text.set_color('white')
        else: text.set_color('black')
    ax_cm.set_title('Confusion Matrix', pad=20, fontsize=36, fontweight='bold')
    ax_cm.set_ylabel('Actual',fontweight='bold')
    ax_cm.set_xlabel('Predicted',fontweight='bold')

    # D. Detailed Metrics Heatmap (Spanning Rows 1 & 2, Col 1)
    ax_metrics = fig.add_subplot(gs[1:, 1])
    rep = classification_report(y_true, y_pred, target_names=classes, output_dict=True)
    metrics_data = [[rep[c]['precision'], rep[c]['recall'], rep[c]['f1-score']] for c in classes]
    df_metrics = pd.DataFrame(metrics_data, columns=['Prec', 'Recall', 'F1'])

    cmap_dark = sns.dark_palette(MAIN_RED, as_cmap=True, input='rgb')
    sns.heatmap(df_metrics, annot=True, fmt='.2f', cmap=cmap_dark, ax=ax_metrics, cbar=False,
                annot_kws={"size": 36, "weight": "bold", "color": "white"})

    ax_metrics.xaxis.tick_top()
    ax_metrics.set_xticklabels(df_metrics.columns, fontsize=36, fontweight='bold')
    ax_metrics.set_yticks([])

    # Giant Vertical Class Labels
    for i, cls in enumerate(classes):
        # Adjusted position for the tall vertical layout
        ax_metrics.text(-0.1, i + 0.5, cls.upper(), fontsize=36, fontweight='bold',
                        rotation=90, va='center', ha='right', transform=ax_metrics.transData)

    # E. Confidence Distribution (Row 1, Col 2)
    ax_hist = fig.add_subplot(gs[1, 2])
    max_probs = np.max(y_prob, axis=1)
    correct_mask = (y_pred == y_true)
    bins = np.linspace(0.3, 1, 15)

    ax_hist.hist(max_probs[~correct_mask], bins=bins, alpha=0.6, color='black',  label='Wrong',   edgecolor='white')
    ax_hist.hist(max_probs[correct_mask],  bins=bins, alpha=0.6, color=MAIN_RED, label='Correct', edgecolor='white')

    ax_hist.set_title('Confidence Distribution', pad=15, fontsize=36, fontweight='bold')
    ax_hist.set_xlabel('Confidence Score',fontweight='bold')
    #ax_hist.set_yticks([])
    ax_hist.legend(frameon=False, loc='upper left', fontsize=24)
    sns.despine(ax=ax_hist, left=True)

    # F. Loss Graph (Row 2, Col 0)
    ax_loss = fig.add_subplot(gs[2, 0])
    loss_hist = history['train_loss']
    loss_epochs = np.arange(1, len(loss_hist) + 1)
    plot_gradient_line(ax_loss, loss_epochs, loss_hist, 'Train Loss', rising_is_good=False, is_loss=True)

    # G. Precision-Recall Curves (Row 2, Col 2)
    ax_pr = fig.add_subplot(gs[2, 2])
    for i in range(n_classes):
        if np.sum(y_bin[:, i]) > 0:
            precision, recall, _ = precision_recall_curve(y_bin[:, i], y_prob[:, i])
            ap = average_precision_score(y_bin[:, i], y_prob[:, i])
            ax_pr.plot(recall, precision, lw=4, color=CLS_COLORS[i % 4],
                       label=f'{classes[i]} ({ap:.2f})')
    ax_pr.set_xlim(-0.02, 1.02)
    ax_pr.set_ylim(-0.02, 1.02)
    ax_pr.set_title('Precision-Recall Curves', fontsize=36, fontweight='bold')
    ax_pr.set_xlabel('Recall')
    ax_pr.set_ylabel('Precision')
    ax_pr.legend(loc='lower left', frameon=False, fontsize=25)

    # Save
    logger.save_figure(fig, "full_report.png")
    plt.close(fig)
    return model, loader, classes, history, 0

# SUBMISSION

In [11]:
class TestDataset(torch.utils.data.Dataset):
    def __init__(self, root, transform):
        self.files = list(Path(root).glob("*.*"))
        self.transform = transform
    def __len__(self): return len(self.files)
    def __getitem__(self, idx):
        path = self.files[idx]
        img = Image.open(path).convert('RGB')
        return self.transform(img), path.name

def create_submission(model, classes, logger, config):
    test_tf = transforms.Compose([
        transforms.Resize((config['img_size'], config['img_size'])),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    ds = TestDataset(config['test_dir'], test_tf)
    if len(ds) == 0:
        print("Skipping submission: No files found.")
        return

    loader = DataLoader(ds, batch_size=config['batch_size'], shuffle=False)
    results = []

    model.eval()
    print("Generating Submission...")
    with torch.no_grad():
        for x, fnames in loader:
            x = x.to(config['device'])
            _, preds = torch.max(model(x), 1)
            for f, p in zip(fnames, preds):
                results.append({'filename': f, 'class': classes[p.item()]})

    df = pd.DataFrame(results)
    logger.save_csv(df, "submission.csv")

# MAIN

In [12]:
def free_gpu():
    if 'model' in globals(): del globals()['model']
    if 'optimizer' in globals(): del globals()['optimizer']
    if 'scheduler' in globals(): del globals()['scheduler']
    gc.collect()
    torch.cuda.empty_cache()
free_gpu()

In [13]:
CONFIG = {
    'repo_url': 'github.com/kvdep/DL-TESTWORK.git',
    'token': token,
    'data_dir': f'{path_filtered}/dataset_filtered',
    'test_dir': f'{path_sub}/test/test/images',
    'batch_size': 32,
    'img_size' : 338,
    'epochs': 5,
    'lr': 1e-3,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu'
}
# Usage inside your script before creating dataloaders:
clean_duplicates(CONFIG['data_dir'], threshold=5)

model, val_loader, class_names, history, best_acc = train(CONFIG)

logger = GitLogger(CONFIG['repo_url'], CONFIG['token'])

logger.start_experiment(best_acc)
logger.save_code()      

run_full_evaluation(model, val_loader, logger, history, class_names, CONFIG)

create_submission(model, class_names, logger, CONFIG)

logger.push_results()

2025-12-18 15:20:54,985: INFO Start: Calculating hashes...


Scanning for duplicates in datasets\datasets\kvdep1\dumplings\versions\5/dataset_filtered...


100%|██████████| 1662/1662 [00:19<00:00, 84.45it/s] 
2025-12-18 15:21:16,754: INFO End: Calculating hashes!
c:\Users\jack_\MAIN\Programms\Python 3.11.9\Lib\site-packages\imagededup\methods\hashing.py:317: RuntimeWarning: Parameter num_enc_workers has no effect since encodings are already provided
  warnings.warn('Parameter num_enc_workers has no effect since encodings are already provided', RuntimeWarning)
2025-12-18 15:21:16,764: INFO Start: Evaluating hamming distances for getting duplicates
2025-12-18 15:21:16,764: INFO Start: Retrieving duplicates using BKTree algorithm
100%|██████████| 1662/1662 [00:14<00:00, 112.44it/s]
2025-12-18 15:21:33,688: INFO End: Retrieving duplicates using BKTree algorithm
2025-12-18 15:21:33,688: INFO End: Evaluating hamming distances for getting duplicates


Found 71 duplicates to remove.
Cleaning finished.
Windows detected: Using 4
Starting Training...


Epoch 1:   0%|          | 0/40 [00:20<?, ?it/s]

...

Epoch 3 | Loss: 0.8603 | Val Acc: 0.8307


Epoch 4:   0%|          | 0/40 [00:21<?, ?it/s]

Epoch 4 | Loss: 0.7014 | Val Acc: 0.9248


Epoch 5:   0%|          | 0/40 [00:20<?, ?it/s]

Epoch 5 | Loss: 0.6108 | Val Acc: 0.9561
[GitLogger] Cleaning up old directory: c:\Users\jack_\MAIN\ВУЗ\DL\КР2\DL-TESTWORK
[GitLogger] Cloning metadata only (Safe Mode)...
[GitLogger] Ready. Target: c:\Users\jack_\MAIN\ВУЗ\DL\КР2\DL-TESTWORK\dumplings\2025-12-18_15-54-11
[GitLogger] Updated Folder Name: 0.9561_2025-12-18_15-54-11
Running Full Evaluation...
Generating Submission...
[GitLogger] Pushing new data...
[GitLogger] Success.
